In [ ]:
#DEPENDENCIES

import os
import torch
import numpy as np
from torch.nn.functional import adaptive_avg_pool2d
from torchvision import transforms
from torchvision.models import inception_v3
import lpips
from torch_fidelity import calculate_metrics
from geomloss import SamplesLoss

In [ ]:
#CONFIGURATION 
#datasets must be in the form (N, C, H, W), torchtensor
dataset_files = [r"path_dataset_1",
                 r"path_dataset_2",
                 r"path_dataset_3"]
output_dirs = ["path_fid", "path_mmd", "path_lpips", "path_kid"] 
batch_size = 32  #lower if needed, number of embeddings calculated per iterations

for d in output_dirs:          #create the folders
    os.makedirs(d, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu" #usage of accellerator

In [ ]:
#FUCTIONS to calculate METRICS of interest, to be able to compute locally the functions works on small batches to be lighter

def load_dataset(path):
    data = torch.load(path)
    if isinstance(data, list):
        data = torch.stack(data)
    return data.float().to(device) / 255.0  # normalize [0,1]



#Maximum Mean Discrepancy, kernel is RBF
def batched_mmd(x, y, batch_size=32, sigma=1.0):   
    n_x, n_y = x.size(0), y.size(0)
    mmd_total = 0.0
    n_batches = max(n_x // batch_size, 1)

    for i in range(0, n_x, batch_size):
        x_batch = x[i:i + batch_size].view(-1, x.size(1) * x.size(2) * x.size(3))
        for j in range(0, n_y, batch_size):
            y_batch = y[j:j + batch_size].view(-1, y.size(1) * y.size(2) * y.size(3))

            xx = torch.cdist(x_batch, x_batch, p=2) ** 2
            yy = torch.cdist(y_batch, y_batch, p=2) ** 2
            xy = torch.cdist(x_batch, y_batch, p=2) ** 2

            k_xx = torch.exp(-xx / (2 * sigma ** 2)).mean()
            k_yy = torch.exp(-yy / (2 * sigma ** 2)).mean()
            k_xy = torch.exp(-xy / (2 * sigma ** 2)).mean()

            mmd_total += (k_xx + k_yy - 2 * k_xy)

    return mmd_total / n_batches

#Learned Perceptual Image Patch Similarity
def batched_lpips(x, y, batch_size=32):
    loss_fn = lpips.LPIPS(net='alex').to(device)
    lpips_values = []
    n_samples = min(len(x), len(y))
    for i in range(0, n_samples, batch_size):
        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]
        for xb, yb in zip(x_batch, y_batch):
            lpips_values.append(loss_fn(xb.unsqueeze(0), yb.unsqueeze(0)).item())
    return float(np.mean(lpips_values))

#FID and KID, Frechet Inception Distance and Kernel Inception Distance
def compute_fid_and_kid(x, y):
    x_uint8 = (x * 255).byte().cpu()
    y_uint8 = (y * 255).byte().cpu()

    metrics = calculate_metrics(
        input1=x_uint8,
        input2=y_uint8,
        cuda=torch.cuda.is_available(),
        isc=False,
        fid=True,
        kid=True,
        verbose=False
    )
    return metrics["frechet_inception_distance"], metrics["kernel_inception_distance_mean"]


In [ ]:
datasets = [load_dataset(f) for f in dataset_files]

for i in range(len(datasets)):
    for j in range(i + 1, len(datasets)):
        name = f"{os.path.basename(dataset_files[i]).split('.')[0]}_vs_{os.path.basename(dataset_files[j]).split('.')[0]}"
        x, y = datasets[i], datasets[j]

        # MMD
        mmd_val = batched_mmd(x, y, batch_size=batch_size)
        with open(f"mmd/{name}.txt", "w") as f:
            f.write(f"{mmd_val}\n")

        # LPIPS
        lpips_val = batched_lpips(x, y, batch_size=batch_size)
        with open(f"lpips/{name}.txt", "w") as f:
            f.write(f"{lpips_val}\n")
            
        # FID & KID 
        fid_val, kid_val = compute_fid_and_kid(x, y)
        with open(f"fid/{name}.txt", "w") as f:
            f.write(f"{fid_val}\n")
        with open(f"kid/{name}.txt", "w") as f:
            f.write(f"{kid_val}\n")

        print(f"Computed metrics for {name}")